# 50. Fairness, privacy, and responsible AI evaluation

**Level:** Intermediate to advanced

**Prerequisites:** 20 and 43: metrics and reliability.

[Course index](00_START_HERE.ipynb) | [Previous](49_MLOps_Drift_and_Model_Artifacts.ipynb) | [Next](51_Time_Series_and_Forecasting.ipynb)

**Topics:** Bias, Fairness, Privacy, Responsible AI, Model cards, Data provenance, Safety.

**Matching script:** `examples\50_Fairness_Privacy_and_Responsible_AI.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Statistical model bias and social unfairness are related but different concepts. A model can
have low overall error while producing unequal error rates for relevant groups. Compare
subgroup sample counts, uncertainty, data quality, and deployment impact rather than drawing
conclusions from an aggregate metric alone.

Fairness definitions can conflict when base rates differ. Equal opportunity examines true
positive rates across groups; demographic parity examines positive prediction rates.
Neither is a universal legal or ethical compliance certificate. Domain stakeholders must
define appropriate objectives and review tradeoffs.

Privacy includes minimizing collection, restricting access, limiting retention, preventing
unintended logging, and reviewing data provenance. Redaction by a few regexes is incomplete;
differential privacy requires a defined mechanism and privacy accounting, not adding arbitrary
noise. Never train on confidential operational payloads without authorization.

A model card documents intended use, exclusions, dataset limitations, evaluation slices,
failure modes, and monitoring/rollback plans. Human review is a designed responsibility, not
a generic checkbox that makes an unreliable model safe.

## Mathematics or data contract

Group true-positive rate = true positives in the group / actual positives in the group; undefined denominators need explicit treatment.

## Important limitation

The exercise uses synthetic site groups, not personal or protected-characteristic data, and is not legal guidance.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Expose a subgroup failure hidden by one total

**Purpose:** Compute denominators and rates transparently.

### Read the code in this order
1. Create synthetic outcomes for two sites.
2. Calculate per-site recall only where positive examples exist.
3. Retain group counts.
4. Assert the unequal rates and avoid labeling the model compliant.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
overall accuracy: 0.875
subgroup evidence: {np.str_('a'): {'samples': 4, 'actual_positives': 2, 'recall': 1.0}, np.str_('b'): {'samples': 4, 'actual_positives': 2, 'recall': 0.5}}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

truth = np.array([1, 1, 0, 0, 1, 1, 0, 0])
prediction = np.array([1, 1, 0, 0, 1, 0, 0, 0])
sites = np.array(["a"] * 4 + ["b"] * 4)
report = {}
for site in sorted(set(sites)):
    mask = sites == site
    positives = int(((truth == 1) & mask).sum())
    true_positives = int(((truth == 1) & (prediction == 1) & mask).sum())
    recall = None if positives == 0 else true_positives / positives
    report[site] = {"samples": int(mask.sum()), "actual_positives": positives, "recall": recall}
assert report["a"]["recall"] == 1.0 and report["b"]["recall"] == 0.5
print("overall accuracy:", float((truth == prediction).mean()))
print("subgroup evidence:", report)

overall accuracy: 0.875
subgroup evidence: {np.str_('a'): {'samples': 4, 'actual_positives': 2, 'recall': 1.0}, np.str_('b'): {'samples': 4, 'actual_positives': 2, 'recall': 0.5}}


### Interpret example 1

The result should demonstrate this purpose: **Compute denominators and rates transparently.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add subgroup sample counts and a zero-positive group; report unknown rather than inventing a zero recall.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Bias` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Group true-positive rate = true positives in the group / actual positives in the group; undefined denominators need explicit treatment.
- Name the boundary of the demonstration: The exercise uses synthetic site groups, not personal or protected-characteristic data, and is not legal guidance.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`
- `https://www.nist.gov/itl/ai-risk-management-framework`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.